# Faster R-CNN 1차 실험 결과 분석

**작업 순서:** 흐름 정리 → 이 노트북에서 설명·표·그래프 작성 → 노션 최종 정리 → Google Docs 변환.

저장된 **seed 0·1·2의 로그와 각자 best checkpoint의 validation 예측**을 분석한다. 모델 학습·추론은 실행하지 않는다. 설정은 CSV 대신 이 노트북과 노션에서 표로 정리한다.

## 목차
1. 데이터 범위와 실험 설정
2. 학습 과정: 충분히 학습했는가?
3. 클래스 성능: 어떤 클래스를 잘 구분했는가?
4. 박스 분석: 위치를 얼마나 정밀하게 맞췄는가?
5. 오류 분석: 못 맞춘 사례는 무엇이었는가?
6. 정리 및 해석

## 0. 분석 목적과 기준
- 전체 232장 중 **train 185장 / validation 47장**이다.
- 학습 로그는 100 epoch 전체를 확인한다. Last와 best의 별도 성능 비교표는 만들지 않는다.
- 예측 분석 대상은 seed 0·1·2의 **best epoch 91·90·81**이다.
- **혼동행렬·precision·recall: score ≥ 0.30, 매칭 IoU ≥ 0.75.** 0.30은 이번 보고서에서 사용자가 정한 진단 기준이다.
- **AP: IoU 0.75·0.80·0.85·0.90·0.95**, maxDets=100. 저장된 추론 score 0.001을 유지하며 진단용 0.30 필터를 적용하지 않는다.
- 클래스와 박스 위치가 함께 맞아야 AP에서 정답이다. AP를 순수한 박스 위치 지표로 해석하지 않는다.
- **1차 seed 0·1·2의 Kaggle 제출을 완료했다.** 실제 제출 점수와 validation 비교는 6.3에 정리한다.

**실행 안내:** 기존 프로젝트 패키지가 준비된 Python 커널에서 위에서 아래로 실행한다. 경로 셀만 실행 환경에 맞게 바꾸며, 패키지는 자동 설치하지 않는다. 그림은 기본적으로 화면에만 표시한다. 표·그래프 실행 결과가 노트북에 저장돼 있다.

In [ ]:
# 분석에 필요한 패키지만 불러온다. 모델 생성·가중치 복원·추론은 하지 않는다.
import contextlib
import csv
import hashlib
import io
import json
import math
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager, patches
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import Markdown, display
from torchmetrics.detection.mean_ap import MeanAveragePrecision

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 80)
for candidate in ("AppleGothic", "NanumGothic", "Malgun Gothic", "Noto Sans CJK KR"):
    if candidate in {font.name for font in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = candidate
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120

# 로컬 기본값. Colab 등에서는 이미 연결된 Drive의 실제 경로로 직접 수정한다.
REPO_ROOT = Path("/Users/codeit/Desktop/codeit/Project1/pill-object-detection")
DRIVE_BASE = Path("/Users/codeit/Library/CloudStorage/GoogleDrive-khsbear0115@gmail.com/My Drive/코드잇 ;)/Project1_pill-object-detection/pill data")
RUN_GROUP = DRIVE_BASE / "cj_runs/exp1_b16_e100/84c1a9494dfd"
RUN_NAMES = {
    0: "cj_faster_e100_s0_exp1_b16_fast_e100_ft_lr3e-4__88800b754961",
    1: "cj_faster_e100_s1_exp1_b16_fast_e100_ft_lr3e-4__91b0ad1f9226",
    2: "cj_faster_e100_s2_exp1_b16_fast_e100_ft_lr3e-4__b013ae09ae90",
}
RUN_DIRS = {seed: RUN_GROUP / name for seed, name in RUN_NAMES.items()}
SEEDS = tuple(sorted(RUN_DIRS))
DIAGNOSTIC_SCORE = 0.30
DIAGNOSTIC_IOU = 0.75
AP_IOUS = [0.75, 0.80, 0.85, 0.90, 0.95]
SAVE_FIGURES = False
OUTPUT_DIR = REPO_ROOT / "notebooks/modeling/exp1_report_figures"
SEED_COLORS = {0: "#2563eb", 1: "#d97706", 2: "#059669"}
IOU_COLORS = ["#2563eb", "#d97706", "#059669", "#9333ea", "#dc2626"]
IMAGE_ROOTS = [
    REPO_ROOT / "data/raw/train_images",
    DRIVE_BASE.parent / "data/raw/train_images",
]
# 필요하면 원본 파일명 → 현재 이미지 경로를 직접 지정한다.
IMAGE_PATH_OVERRIDES = {}

In [ ]:
def read_json(file):
    """지정한 JSON 하나만 읽는다."""
    return json.loads(Path(file).read_text(encoding="utf-8-sig"))


def file_sha256(file):
    """원본을 바꾸지 않고 SHA-256을 계산한다."""
    digest = hashlib.sha256()
    with Path(file).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def kcode(category):
    return "미매칭" if category is None else f"K-{int(category):06d}"


def format_value(value):
    if value is None:
        return "미확인"
    if isinstance(value, (list, dict, tuple)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)


def show_table(frame, title=None):
    """추가 표 패키지 없이 복사 가능한 Markdown 표를 표시한다."""
    if title:
        display(Markdown(f"**{title}**"))
    if frame.empty:
        display(Markdown("해당 조건에 해당하는 항목이 없습니다."))
        return
    def clean(value):
        if value is None or (isinstance(value, float) and math.isnan(value)):
            return "—"
        return format_value(value).replace("|", r"\|").replace("\n", " ")
    lines = ["| " + " | ".join(map(str, frame.columns)) + " |",
             "| " + " | ".join(["---"] * len(frame.columns)) + " |"]
    lines += ["| " + " | ".join(clean(v) for v in row) + " |"
              for row in frame.itertuples(index=False, name=None)]
    display(Markdown("\n".join(lines)))


def show_figure(fig, name):
    """기본은 화면 표시이며 명시적으로 선택할 때만 PNG를 저장한다."""
    fig.tight_layout()
    if SAVE_FIGURES:
        output = OUTPUT_DIR.resolve()
        if not output.is_relative_to(REPO_ROOT.resolve()) or output.is_relative_to(DRIVE_BASE.resolve()):
            raise ValueError("출력은 공유 Drive 밖의 현재 저장소 내부에 지정하세요.")
        output.mkdir(parents=True, exist_ok=True)
        target = output / (name + ".png")
        if target.exists():
            raise FileExistsError(f"기존 그림을 덮어쓰지 않습니다: {target}")
        fig.savefig(target, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)


required = [RUN_GROUP / "settings.json", RUN_GROUP / "class_support.csv"]
required += [root / file for root in RUN_DIRS.values()
             for file in ("config.json", "result.json", "metrics.csv",
                          "comparison_manifest.json", "validation_predictions.pt")]
missing = [str(file) for file in required if not file.is_file()]
if missing:
    raise FileNotFoundError("필수 자료가 없습니다. 경로 셀을 수정하세요:\n" + "\n".join(missing))

SETTINGS = read_json(RUN_GROUP / "settings.json")
RUNS = {}
for seed, root in RUN_DIRS.items():
    config = read_json(root / "config.json")
    result = read_json(root / "result.json")
    manifest = read_json(root / "comparison_manifest.json")
    history = pd.read_csv(root / "metrics.csv")
    if config["spec"]["seed"] != seed or result["seed"] != seed:
        raise ValueError(f"seed {seed}: 지정 폴더의 seed가 다릅니다.")
    epochs = history["epoch"].to_numpy()
    if len(epochs) == 0 or not np.array_equal(epochs, np.arange(1, int(result["last_epoch"]) + 1)):
        raise ValueError(f"seed {seed}: epoch 기록이 비었거나 중복·누락됐습니다.")
    numeric = history.select_dtypes(include=np.number)
    if not np.isfinite(numeric.to_numpy()).all():
        raise ValueError(f"seed {seed}: 로그에 NaN 또는 무한대가 있습니다.")
    best = history.loc[history["epoch"].eq(result["best_epoch"])].iloc[0]
    if not math.isclose(best["val_map_75_95"], result["best_val_map_75_95"], abs_tol=1e-6):
        raise ValueError(f"seed {seed}: best epoch의 로그와 result 점수가 다릅니다.")
    if not math.isclose(history["val_map_75_95"].max(), result["best_val_map_75_95"], abs_tol=1e-6):
        raise ValueError(f"seed {seed}: 로그의 최고점과 result의 best 점수가 다릅니다.")
    RUNS[seed] = dict(root=root, config=config, result=result, manifest=manifest, history=history)
if len({r["config"]["recipe"]["data_identity"] for r in RUNS.values()}) != 1:
    raise ValueError("세 seed의 학습 데이터 식별자가 다릅니다.")
print("입력 확인 완료: seed 0·1·2. 로그와 저장된 validation 예측만 사용합니다.")

## 1. 데이터 범위와 실험 설정

### 1.1 데이터 구성
- 전체 232장 = train 185장 + validation 47장.
- Train 612객체·56클래스 / validation 159객체·30클래스.
- 나머지 학습 클래스 26개는 validation 정답이 없어서 AP 평가 대상이 아니다.
- Train에 없고 test에 등장하는 18개 클래스는 초기 모델이 학습하지 못했다.
- Test 빈도는 프로젝트의 이미지 검토 추정치이며 공식 test GT가 아니다.

아래 코드는 저장된 표와 분할 목록을 읽어 숫자를 다시 확인한다. 라벨을 다시 만들거나 수정하지 않는다.

In [ ]:
SUPPORT = pd.read_csv(RUN_GROUP / "class_support.csv")
for column in ("category_id", "train", "val"):
    SUPPORT[column] = SUPPORT[column].astype(int)
if SUPPORT["category_id"].duplicated().any():
    raise ValueError("클래스 빈도표에 중복 category_id가 있습니다.")
if (SUPPORT[["train", "val"]] < 0).any().any():
    raise ValueError("클래스 빈도표에 음수 객체 수가 있습니다.")
CLASS_IDS = sorted(SUPPORT["category_id"].tolist())
CLASS_NAMES = SUPPORT.set_index("category_id")["name"].to_dict()
FREQUENCY = SUPPORT.set_index("category_id")["train"].to_dict()
VAL_FREQUENCY = SUPPORT.set_index("category_id")["val"].to_dict()
counts = SETTINGS["split_verification"]["counts"]
show_table(pd.DataFrame([
    ["전체 이미지", sum(counts.values()), "settings.json / split_verification"],
    ["Train 이미지", counts["train"], "고정 분할"],
    ["Validation 이미지", counts["val"], "고정 분할"],
    ["Train 객체", int(SUPPORT["train"].sum()), "class_support.csv"],
    ["Validation 객체", int(SUPPORT["val"].sum()), "class_support.csv"],
    ["Train 정답 클래스", int(SUPPORT["train"].gt(0).sum()), "class_support.csv"],
    ["Validation 정답 클래스", int(SUPPORT["val"].gt(0).sum()), "class_support.csv"],
    ["분할 seed", SETTINGS["split_verification"]["split_seed"], "저장된 분할 검증 기록"],
], columns=["항목", "값", "근거"]))
no_val = SUPPORT.loc[SUPPORT["val"].eq(0), ["category_id", "name", "train", "val"]].copy()
no_val.insert(1, "k_code", no_val["category_id"].map(kcode))
no_val["AP 평가"] = "validation 정답 없음"
show_table(no_val, "Validation에서 AP를 평가할 수 없는 학습 클래스")
classes_path = REPO_ROOT / "data/datasets/classes.csv"
if classes_path.is_file():
    project_classes = pd.read_csv(classes_path)
    test_only = project_classes.loc[project_classes["group"].eq("test_only"),
                                   ["k_code", "name", "group"]]
    show_table(test_only, "초기 train에 없던 test 등장 클래스 (프로젝트 검토 기준)")
else:
    display(Markdown("test 전용 18개 클래스 목록은 classes.csv 경로 확인 후 표시합니다."))

### 1.2 클래스별 데이터 분포

56개 학습 클래스 전체의 **정답 객체 수**를 비교한다. 학습 빈도는 이미지 수나 epoch 동안 반복해서 본 누적 횟수가 아니다.

In [ ]:
ordered_support = SUPPORT.sort_values(["train", "category_id"])
frequency_table = ordered_support.assign(k_code=ordered_support["category_id"].map(kcode))
show_table(frequency_table[["k_code", "name", "train", "val"]], "전체 클래스 객체 수")
fig, ax = plt.subplots(figsize=(13, max(10, len(ordered_support) * 0.25)))
y = np.arange(len(ordered_support))
ax.barh(y - 0.18, ordered_support["train"], height=0.34, label="Train 객체", color="#2563eb")
ax.barh(y + 0.18, ordered_support["val"], height=0.34, label="Validation 객체", color="#059669")
ax.set_yticks(y, [f"{kcode(r.category_id)} {r.name}" for r in ordered_support.itertuples()], fontsize=8)
ax.invert_yaxis()
ax.set(xlabel="정답 객체 수 (개)", title="1차 데이터: 클래스별 train·validation 객체 수")
ax.legend()
show_figure(fig, "01_class_frequency")
positive = SUPPORT.loc[SUPPORT["train"].gt(0), "train"]
show_table(pd.DataFrame([{
    "최소 객체 수": int(positive.min()), "최대 객체 수": int(positive.max()),
    "최대/최소": round(float(positive.max() / positive.min()), 3),
    "중앙값": float(positive.median()),
}]), "학습 객체 수의 불균형 요약")

### 1.3 실험 설정표

세 seed의 **저장된 설정 키 전체**를 표로 표시한다. 학습 batch는 로그로 실제 적용값을 함께 확인한다. 추론 batch처럼 config에 없는 값은 현재 코드에서 추정해 채우지 않고 근거의 범위를 표시한다.

학습 중 미리보기 score 0.30은 과거 저장 설정이다. 이번 진단 score 0.30이나 AP 추론 score 0.001과 구분한다.

### 저장 설정 요약표 (작성 시 원본 대조)

| 항목 · 설정 키 | 세 seed의 공통 저장값 |
| --- | --- |
| 최대 epoch · `max_epochs` | 100 |
| 조기 종료 대기 epoch · `patience` | 30 |
| 조기 종료 사용 · `early_stopping` | true |
| 입력 크기 설정 · `image_size` | 640 |
| Train batch 설정 · `batch_size` | 16 |
| 기울기 누적 · `accumulation_steps` | 1 |
| Weight decay · `weight_decay` | 0.0001 |
| AdamW betas · `betas` | [0.9,0.999] |
| Warmup epoch · `warmup_epochs` | 3 |
| 최종 LR 비율 · `final_lr_ratio` | 0.01 |
| DataLoader workers · `num_workers` | 2 |
| AP 추론 score · `score_threshold` | 0.001 |
| NMS IoU · `nms_iou` | 0.5 |
| 이미지당 최대 검출 · `max_detections` | 100 |
| 학습 중 미리보기 score · `preview_score` | 0.3 |
| 학습 중 미리보기 수 · `preview_count` | 3 |
| 실험 태그 · `experiment_tag` | exp1_b16_fast_e100 |
| W&B 프로젝트 · `wandb_project` | pill-object-detection |
| W&B 엔티티 · `wandb_entity` | codeit-prj1-team2 |
| W&B 기록 모드 · `wandb_mode` | online |
| W&B 사용 · `use_wandb` | true |
| 학습 CUDA 요구 · `require_cuda` | true |
| Deterministic 설정 · `deterministic` | false |
| 학습 정밀도 · `precision` | amp_fp16 |
| 정규화 · `normalization` | FrozenBatchNorm2d_eps0_fixed_statistics_and_affine |
| 증강 · `augmentation` | none |
| 사전학습 가중치 · `weights` | COCO_V1 |
| Optimizer · `optimizer` | AdamW |
| 기본 학습률 · `learning_rate` | 0.0003 |
| 학습률 목록 · `learning_rates` | [0.0003] |
| Validation loss batch · `val_loss_batch_size` | 1 |
| Validation loss 측정 seed · `loss_measurement_seed` | 0 |
| 추론 정밀도 · `inference_precision` | fp32 |
| Validation loss 정밀도 · `val_loss_precision` | fp32 |

학습 seed만 0·1·2로 다르다. 아래 코드는 지정한 원본에서 전체 표를 다시 표시하고 실제 로그와 대조한다. 과거 미리보기 score 0.30은 수정하지 않고, 이번 오류 진단은 0.30으로 별도 적용한다.

In [ ]:
SETTING_LABELS = {
    "max_epochs": "최대 epoch", "patience": "조기 종료 대기 epoch", "early_stopping": "조기 종료 사용",
    "image_size": "입력 크기 설정", "batch_size": "Train batch 설정", "accumulation_steps": "기울기 누적",
    "weight_decay": "Weight decay", "betas": "AdamW betas", "warmup_epochs": "Warmup epoch",
    "final_lr_ratio": "최종 LR 비율", "num_workers": "DataLoader workers", "score_threshold": "AP 추론 score",
    "nms_iou": "NMS IoU", "max_detections": "이미지당 최대 검출", "preview_score": "학습 중 미리보기 score",
    "preview_count": "학습 중 미리보기 수", "experiment_tag": "실험 태그", "wandb_project": "W&B 프로젝트",
    "wandb_entity": "W&B 엔티티", "wandb_mode": "W&B 기록 모드", "use_wandb": "W&B 사용",
    "require_cuda": "학습 CUDA 요구", "deterministic": "Deterministic 설정", "precision": "학습 정밀도",
    "normalization": "정규화", "augmentation": "증강", "weights": "사전학습 가중치", "optimizer": "Optimizer",
    "learning_rate": "기본 학습률", "learning_rates": "학습률 목록", "val_loss_batch_size": "Validation loss batch",
    "loss_measurement_seed": "Validation loss 측정 seed", "inference_precision": "추론 정밀도",
    "val_loss_precision": "Validation loss 정밀도",
}
configs = {seed: run["config"]["recipe"]["config"] for seed, run in RUNS.items()}
keys = sorted(set().union(*(set(c) for c in configs.values())))
settings_rows = []
for key in keys:
    values = [format_value(configs[seed].get(key)) for seed in SEEDS]
    settings_rows.append([SETTING_LABELS.get(key, key), key, *values,
                          "공통" if len(set(values)) == 1 else "차이 있음"])
show_table(pd.DataFrame(settings_rows, columns=["항목", "설정 키", "seed 0", "seed 1", "seed 2", "비교"]),
           "저장된 config 전체")
run_settings = []
for seed, run in RUNS.items():
    h, c, spec = run["history"], configs[seed], run["config"]["spec"]
    run_settings.append({
        "Seed": seed, "학습 범위": spec["mode"], "학습률": spec["lr"],
        "실제 micro batch": sorted(h["micro_batch_size"].unique().tolist()),
        "실제 누적": sorted(h["accumulation_steps"].unique().tolist()),
        "실제 유효 batch 설정": sorted(h["effective_batch_size"].unique().tolist()),
        "마지막 배치 이미지": sorted(h["last_group_images"].unique().tolist()),
        "Validation loss batch 설정": c.get("val_loss_batch_size"),
        "추론 batch 저장값": c.get("inference_batch_size", "미기록"),
        "총 optimizer step": int(h["optimizer_steps"].sum()),
        "AMP 건너뛴 step": int(h["amp_skipped_steps"].sum()),
    })
show_table(pd.DataFrame(run_settings), "Seed별 실제 로그와 저장 설정")
show_table(pd.DataFrame([
    ["모델", "Faster R-CNN ResNet50 FPN", "recipe.runtime.model"],
    ["입력 리사이즈", "원래 학습 코드의 min_size=max_size=640, 종횡비 유지", "04 노트북 build_model; 640×640 정사각 강제 아님"],
    ["학습 범위", "전체 미세조정 (정규화 고정)", "spec.mode=ft / 저장된 normalization"],
    ["스케줄러", "Optimizer step 단위 선형 warmup + cosine", "04 노트북 make_scheduler / LR 로그"],
    ["Checkpoint 선택", "해당 seed의 validation mAP 최고", "result.json / metrics.csv"],
    ["진단 score", DIAGNOSTIC_SCORE, "이번 보고서에서 사용자 지정"],
    ["진단 IoU", DIAGNOSTIC_IOU, "기존 진단의 일대일 매칭 기준 유지"],
    ["AP IoU", AP_IOUS, "comparison_manifest.json"],
], columns=["항목", "값", "근거"]), "모델·평가 해석에 필요한 설정")
environment = SETTINGS["environment"]
env_rows = [[k, format_value(v)] for k, v in environment.items()
            if k not in {"packages", "uv_lock_sha256", "environment_repository_commit"}]
env_rows += [["package." + k, v] for k, v in sorted(environment.get("packages", {}).items())]
show_table(pd.DataFrame(env_rows, columns=["실행 환경", "저장값"]), "학습 당시 환경")

## 2. 학습 과정: 충분히 학습했는가?

### 2.1 학습이 정상적으로 진행됐는가?
### 2.2 Validation 성능은 어떻게 변했는가?

가로축은 로그에 명시된 **epoch**이다. W&B step을 epoch으로 바꾸지 않는다. 세 seed의 train loss·validation loss·validation mAP를 각각 비교하고, 점선은 해당 seed의 best epoch를 표시한다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
plots = [("train_loss", "Train loss", "Loss"),
         ("val_loss", "Validation loss", "Loss"),
         ("val_map_75_95", "Validation mAP@[.75:.95]", "mAP")]
for ax, (column, title, ylabel) in zip(axes, plots):
    for seed, run in RUNS.items():
        h = run["history"]
        ax.plot(h["epoch"], h[column], color=SEED_COLORS[seed], label=f"seed {seed}")
        epoch = run["result"]["best_epoch"]
        value = h.loc[h["epoch"].eq(epoch), column].iloc[0]
        ax.axvline(epoch, color=SEED_COLORS[seed], alpha=0.3, linestyle="--")
        ax.scatter([epoch], [value], color=SEED_COLORS[seed], s=35, zorder=3)
    ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
    ax.grid(alpha=0.18)
    ax.legend()
axes[2].set_ylim(0, 1.02)
show_figure(fig, "02_learning_curves")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for seed, run in RUNS.items():
    h = run["history"]
    axes[0].plot(h["epoch"], h["last_lr"], label=f"seed {seed}", color=SEED_COLORS[seed])
    axes[1].plot(h["epoch"], h["optimizer_steps"].cumsum(), label=f"seed {seed}", color=SEED_COLORS[seed])
axes[0].set(xlabel="Epoch", ylabel="Epoch 종료 시 LR", title="학습률 변화", yscale="log")
axes[1].set(xlabel="Epoch", ylabel="누적 실제 optimizer step", title="실제 학습량")
for ax in axes:
    ax.grid(alpha=0.18)
    ax.legend()
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda value, position: f"{value:.0e}"))
show_figure(fig, "02_lr_optimizer_steps")

### 2.3 과소적합·과적합 징후가 있는가?

- Train loss가 충분히 감소하는지, validation loss·mAP가 함께 개선되는지 확인한다.
- Train loss가 감소해도 validation이 악화되는 구간이 있는지, 세 seed에서 반복되는지 확인한다.
- Train·val loss는 batch·정밀도·측정 방식이 다르므로 절대값의 차이만으로 판단하지 않는다.
- Train mAP는 기록돼 있지 않으면 만들지 않는다. 적은 loss나 높은 validation AP만으로 과소적합·과적합 부재를 확정하지 않는다.

아래 후반부 요약은 **전체 로그의 마지막 20 epoch 추세**다. Last checkpoint와 best checkpoint의 성능 비교표는 만들지 않는다. 시작·끝 값 하나가 아닌 전체 곡선과 함께 해석한다.

In [ ]:
trend_rows = []
for seed, run in RUNS.items():
    h = run["history"]
    tail = h.tail(min(20, len(h)))
    row = {"Seed": seed, "후반 구간": f"{int(tail.epoch.min())}~{int(tail.epoch.max())}"}
    for column, label in [("train_loss", "Train loss"), ("val_loss", "Val loss"), ("val_map_75_95", "Val mAP")]:
        slope = np.polyfit(tail["epoch"], tail[column], 1)[0]
        row[label + " 기울기/epoch"] = round(float(slope), 7)
    row["AMP 건너뛴 step"] = int(h["amp_skipped_steps"].sum())
    row["유효하지 않은 수치"] = int((~np.isfinite(h.select_dtypes(include=np.number))).to_numpy().sum())
    trend_rows.append(row)
show_table(pd.DataFrame(trend_rows), "후반 학습 추세와 수치 안정성")

components = [("loss_classifier", "분류 loss"), ("loss_box_reg", "박스 회귀 loss"),
              ("loss_objectness", "RPN objectness loss"), ("loss_rpn_box_reg", "RPN 박스 loss")]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for index, (column, label) in enumerate(components):
    for seed, run in RUNS.items():
        h = run["history"]
        for row, prefix in [(0, ""), (1, "val_")]:
            name = prefix + column
            if name in h:
                axes[row, index].plot(h["epoch"], h[name], color=SEED_COLORS[seed], label=f"seed {seed}")
    for row, split in [(0, "Train"), (1, "Validation")]:
        axes[row, index].set(xlabel="Epoch", ylabel="Loss", title=split + " " + label)
        axes[row, index].grid(alpha=0.18)
        axes[row, index].legend()
show_figure(fig, "02_loss_components")

### 2.4 각 seed의 best checkpoint

Best는 각 seed의 validation mAP 최고 checkpoint이며, test에서의 최적 checkpoint를 뜻하지 않는다. 이후 모든 예측 분석은 이 best 결과에 맞춘다.

In [ ]:
best_rows = []
for seed, run in RUNS.items():
    r, h = run["result"], run["history"]
    at_best = h.loc[h["epoch"].eq(r["best_epoch"])].iloc[0]
    best_rows.append({"Seed": seed, "Best epoch": r["best_epoch"],
                      "Best validation mAP": r["best_val_map_75_95"],
                      "Best 시점 train loss": float(at_best["train_loss"]),
                      "Best 시점 val loss": float(at_best["val_loss"]),
                      "Best까지 실제 step": int(h.loc[h["epoch"].le(r["best_epoch"]), "optimizer_steps"].sum())})
BEST_TABLE = pd.DataFrame(best_rows)
show_table(BEST_TABLE.round(6), "Seed별 best 결과")
show_table(pd.DataFrame([{"Seed 수": len(BEST_TABLE),
                          "Best mAP 평균": BEST_TABLE["Best validation mAP"].mean(),
                          "Best mAP 표본 표준편차": BEST_TABLE["Best validation mAP"].std(ddof=1)}]).round(6))

## 3. 클래스 성능: 어떤 클래스를 잘 구분했는가?

### 3.1 혼동행렬의 해석 기준과 저장 예측 확인

- 각 seed의 **best validation 예측 캐시**를 CPU로 읽는다. best.pt 자체는 불러오지 않는다.
- 캐시 해시·run signature·동일 GT 여부와 재계산 mAP를 확인한다.
- **행=정답 / 열=예측**, score ≥ 0.30, IoU ≥ 0.75의 클래스 무관 일대일 매칭이다.
- 클래스가 달라도 박스가 IoU 기준 이상 겹치면 해당 혼동 칸에 집계한다. 박스가 기준을 넘지 못하면 미매칭으로 남는다.
- 이는 순수한 분류 정확도가 아니며 AP의 클래스별 score 순 매칭과도 다르다.

아래 AP 계산은 캐시의 score 0.001 결과를 그대로 사용한다. 혼동행렬에만 score 0.30을 적용한다.

In [ ]:
def cpu_item(item):
    """캐시 Tensor의 dtype과 값을 유지해 CPU 평가 입력을 만든다."""
    result = {}
    for key in ("boxes", "labels", "scores", "area", "iscrowd"):
        if key in item:
            value = item[key]
            if isinstance(value, torch.Tensor):
                result[key] = value.detach().cpu()
            else:
                dtype = torch.int64 if key in {"labels", "iscrowd"} else torch.float32
                result[key] = torch.as_tensor(value, dtype=dtype, device="cpu")
    return result


def calculate_ap(records):
    """학습 당시와 같은 TorchMetrics AP 경로를 사용한다."""
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", iou_thresholds=AP_IOUS,
                                  max_detection_thresholds=[1, 10, 100],
                                  class_metrics=True, extended_summary=True)
    with contextlib.redirect_stdout(io.StringIO()):
        metric.update([cpu_item(r["prediction"]) for r in records],
                      [cpu_item(r["target"]) for r in records])
        measured = metric.compute()
    precision = measured["precision"][:, :, :, 0, -1].detach().cpu().numpy()
    category_ids = measured["classes"].reshape(-1).tolist()
    support = Counter(int(c) for r in records for c in r["target"]["labels"].tolist())
    def valid_mean(values):
        valid = values[values >= 0]
        return float(valid.mean()) if valid.size else np.nan
    rows = []
    for index, category in enumerate(category_ids):
        category = int(category)
        if not support[category]:
            continue
        row = {"category_id": category, "k_code": kcode(category), "name": CLASS_NAMES.get(category, ""),
               "GT_objects": support[category], "train_objects": FREQUENCY.get(category, 0),
               "AP75_95": valid_mean(precision[:, :, index])}
        row.update({f"AP{round(iou * 100)}": valid_mean(precision[t, :, index])
                    for t, iou in enumerate(AP_IOUS)})
        row["AP75_minus_AP95"] = row["AP75"] - row["AP95"]
        rows.append(row)
    return float(measured["map"]), pd.DataFrame(rows)


def validate_records(records):
    """이미지·클래스·좌표·정답 객체 수를 확인한다."""
    names = [r["file_name"] for r in records]
    if len(names) != counts["val"] or len(set(names)) != len(names):
        raise ValueError("Validation 이미지 수 또는 파일명 중복을 확인하세요.")
    support = Counter()
    for record in records:
        for kind in ("target", "prediction"):
            item = record[kind]
            boxes = item["boxes"]
            labels = item["labels"]
            if boxes.ndim != 2 or boxes.shape[1] != 4 or len(boxes) != len(labels):
                raise ValueError(f"{record['file_name']}: {kind} boxes/labels 형식이 다릅니다.")
            if not torch.isfinite(boxes).all() or not (boxes[:, 2:] > boxes[:, :2]).all():
                raise ValueError(f"{record['file_name']}: 유효하지 않은 xyxy 박스입니다.")
            if not set(map(int, labels.tolist())).issubset(CLASS_IDS):
                raise ValueError("캐시 라벨이 원래 category_id 목록과 다릅니다.")
            if kind == "prediction":
                scores = item["scores"]
                if len(scores) != len(boxes) or not torch.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
                    raise ValueError("예측 score 형식이 다릅니다.")
        support.update(map(int, record["target"]["labels"].tolist()))
    expected = {c: n for c, n in VAL_FREQUENCY.items() if n}
    if dict(support) != expected:
        raise ValueError("캐시의 클래스별 GT 객체 수와 class_support.csv가 다릅니다.")


RECORDS, AP_TABLES, AP_CHECKS = {}, {}, []
for seed, run in RUNS.items():
    manifest, config, result = run["manifest"], run["config"], run["result"]
    cache = run["root"] / "validation_predictions.pt"
    if file_sha256(cache) != manifest["prediction_sha256"]:
        raise ValueError(f"seed {seed}: validation 캐시 해시가 manifest와 다릅니다.")
    if manifest["run_signature"] != config["signature"] or manifest["spec"]["seed"] != seed:
        raise ValueError(f"seed {seed}: 캐시가 다른 run에 연결됐습니다.")
    if manifest["result"]["best_epoch"] != result["best_epoch"]:
        raise ValueError(f"seed {seed}: 캐시에 기록된 best epoch가 다릅니다.")
    if list(manifest["iou_thresholds"]) != AP_IOUS or manifest["max_detections"] != 100:
        raise ValueError("저장된 AP 평가 조건이 이번 분석과 다릅니다.")
    if not math.isclose(manifest["score_threshold"], configs[seed]["score_threshold"], abs_tol=1e-12):
        raise ValueError("저장된 AP score 조건이 다릅니다.")
    try:
        payload = torch.load(cache, map_location="cpu", weights_only=True)
    except Exception as error:
        raise RuntimeError("저장 예측을 안전하게 읽지 못했습니다. weights_only=False로 바꾸지 말고 캐시 형식을 확인하세요.") from error
    records = payload.get("records", [])
    for record in records:
        record["target"] = cpu_item(record["target"])
        record["prediction"] = cpu_item(record["prediction"])
    validate_records(records)
    if RECORDS:
        reference = {r["file_name"]: r for r in RECORDS[SEEDS[0]]}
        for record in records:
            original = reference.get(record["file_name"])
            if original is None or any(not torch.equal(record["target"][key], original["target"][key]) for key in ("boxes", "labels")):
                raise ValueError("Seed 사이 validation 정답이 다릅니다.")
    measured, per_class = calculate_ap(records)
    saved = result["best_val_map_75_95"]
    if not math.isclose(measured, saved, rel_tol=0, abs_tol=1e-5):
        raise ValueError(f"seed {seed}: 재계산 mAP={measured:.10f}, 저장 점수={saved:.10f}. 평가 환경을 확인하세요.")
    RECORDS[seed], AP_TABLES[seed] = records, per_class
    AP_CHECKS.append({"Seed": seed, "Best epoch": result["best_epoch"], "이미지": len(records),
                      "저장 mAP": saved, "재계산 mAP": measured, "절대 차이": abs(measured-saved)})
show_table(pd.DataFrame(AP_CHECKS).round(8), "Best 예측 캐시와 AP 일치 확인")

In [ ]:
from torchvision.ops import box_iou

GT_KINDS = {
    "correct": "정답", "class_confusion": "클래스 혼동", "localization_candidate": "위치 불일치 후보",
    "mixed_candidate": "위치·클래스 복합 후보", "unmatched_gt": "미매칭 GT·원인 미분류",
}
PRED_KINDS = {"duplicate": "중복 예측 후보", "extra": "추가 예측·원인 미분류"}


def greedy_pairs(overlaps, gt_ids, pred_ids, minimum, maximum=None):
    """IoU가 큰 순서로 각 GT·예측을 한 번씩 연결한다."""
    candidates = [(float(overlaps[g, p]), g, p) for g in gt_ids for p in pred_ids
                  if (overlaps[g, p] > 0 if minimum == 0 else overlaps[g, p] >= minimum)
                  and (maximum is None or overlaps[g, p] < maximum)]
    used_g, used_p, pairs = set(), set(), []
    for value, g, p in sorted(candidates, key=lambda x: (-x[0], x[1], x[2])):
        if g not in used_g and p not in used_p:
            pairs.append((g, p, value))
            used_g.add(g)
            used_p.add(p)
    return pairs


def diagnose(seed, threshold):
    """0.75 매칭 혼동행렬과 겹치지 않는 GT별 오류 후보 목록을 만든다."""
    positions = {category: i for i, category in enumerate(CLASS_IDS)}
    bg = len(CLASS_IDS)
    matrix = np.zeros((bg + 1, bg + 1), dtype=int)
    gt_events, pred_events = [], []
    for record in RECORDS[seed]:
        target, prediction = record["target"], record["prediction"]
        keep = torch.where(prediction["scores"] >= threshold)[0]
        boxes, labels = prediction["boxes"][keep], prediction["labels"][keep]
        overlaps = box_iou(target["boxes"], boxes).numpy()
        truth = target["labels"].tolist()
        predicted = labels.tolist()
        pairs = greedy_pairs(overlaps, range(len(truth)), range(len(predicted)), DIAGNOSTIC_IOU)
        used_g = {g for g, _, _ in pairs}
        used_p = {p for _, p, _ in pairs}

        def event(kind, g=None, p=None, iou=None):
            return {"Seed": seed, "file_name": record["file_name"], "kind": kind,
                    "gt_index": g, "pred_index": None if p is None else int(keep[p]),
                    "true_category": None if g is None else int(truth[g]),
                    "pred_category": None if p is None else int(predicted[p]),
                    "score": None if p is None else float(prediction["scores"][keep[p]]),
                    "IoU": iou, "threshold": threshold}

        for g, p, value in pairs:
            t, c = int(truth[g]), int(predicted[p])
            matrix[positions[t], positions[c]] += 1
            gt_events.append(event("correct" if t == c else "class_confusion", g, p, value))
        remaining_g = [g for g in range(len(truth)) if g not in used_g]
        remaining_p = [p for p in range(len(predicted)) if p not in used_p]
        for g in remaining_g:
            matrix[positions[int(truth[g])], bg] += 1
        for p in remaining_p:
            matrix[bg, positions[int(predicted[p])]] += 1

        # AP나 혼동행렬을 바꾸지 않는 사례 탐색용 보조 규칙이다.
        # 남은 박스가 IoU 0 초과·0.75 미만일 때 클래스와 무관하게 일대일 연결한다.
        low_pairs = greedy_pairs(overlaps, remaining_g, remaining_p, 0.0, DIAGNOSTIC_IOU)
        candidate_g = {g for g, _, _ in low_pairs}
        candidate_p = {p for _, p, _ in low_pairs}
        for g, p, value in low_pairs:
            kind = "localization_candidate" if truth[g] == predicted[p] else "mixed_candidate"
            gt_events.append(event(kind, g, p, value))
        for g in remaining_g:
            if g not in candidate_g:
                gt_events.append(event("unmatched_gt", g))
        for p in remaining_p:
            if p in candidate_p:
                continue
            duplicate = [g for g in range(len(truth))
                         if truth[g] == predicted[p] and overlaps[g, p] >= DIAGNOSTIC_IOU]
            g = max(duplicate, key=lambda g: overlaps[g, p]) if duplicate else None
            pred_events.append(event("duplicate" if g is not None else "extra", g, p,
                                     None if g is None else float(overlaps[g, p])))
    gt_frame = pd.DataFrame(gt_events)
    pred_frame = pd.DataFrame(pred_events, columns=gt_frame.columns)
    if len(gt_frame) != int(SUPPORT["val"].sum()):
        raise ValueError("GT별 진단 목록의 객체 수가 맞지 않습니다.")
    if gt_frame.duplicated(["file_name", "gt_index"]).any():
        raise ValueError("같은 GT가 오류 후보 목록에 중복됐습니다.")
    tp = int(np.trace(matrix[:-1, :-1]))
    fp = int(matrix[:, :-1].sum() - tp)
    fn = int(matrix[:-1, :].sum() - tp)
    summary = {"Seed": seed, "score": threshold, "매칭 IoU": DIAGNOSTIC_IOU,
               "TP": tp, "FP": fp, "FN": fn,
               "Precision": tp / (tp + fp) if tp + fp else np.nan,
               "Recall": tp / (tp + fn) if tp + fn else np.nan}
    per_class = []
    pairs_rows = []
    for i, category in enumerate(CLASS_IDS):
        ctp = int(matrix[i, i])
        cfp = int(matrix[:, i].sum() - ctp)
        cfn = int(matrix[i, :].sum() - ctp)
        support = ctp + cfn
        per_class.append({"Seed": seed, "k_code": kcode(category), "name": CLASS_NAMES[category],
                          "GT 객체": support, "TP": ctp, "FP": cfp, "FN": cfn,
                          "Precision": ctp / (ctp + cfp) if ctp + cfp else np.nan,
                          "Recall": ctp / support if support else np.nan})
        for j, predicted_category in enumerate(CLASS_IDS):
            count = int(matrix[i, j])
            if i != j and count:
                pairs_rows.append({"Seed": seed, "정답 코드": kcode(category), "정답 클래스": CLASS_NAMES[category],
                                   "예측 코드": kcode(predicted_category), "예측 클래스": CLASS_NAMES[predicted_category],
                                   "혼동 건수": count, "정답 객체 수": support, "혼동 비율": count / support})
    pair_columns = ["Seed", "정답 코드", "정답 클래스", "예측 코드", "예측 클래스", "혼동 건수", "정답 객체 수", "혼동 비율"]
    pair_frame = pd.DataFrame(pairs_rows, columns=pair_columns)
    if not pair_frame.empty:
        pair_frame = pair_frame.sort_values(["혼동 건수", "정답 코드", "예측 코드"], ascending=[False, True, True])
    return {"matrix": matrix, "summary": summary, "per_class": pd.DataFrame(per_class),
            "pairs": pair_frame, "gt_events": gt_frame, "pred_events": pred_frame}


DIAGNOSTICS = {seed: diagnose(seed, DIAGNOSTIC_SCORE) for seed in SEEDS}
DIAGNOSTIC_TABLE = pd.DataFrame([d["summary"] for d in DIAGNOSTICS.values()])
show_table(DIAGNOSTIC_TABLE.round(6), "score 0.30 / IoU 0.75 / micro 집계")

### 3.2 Seed별 전체 혼동행렬과 오류 중심 혼동행렬

각 seed마다 **전체 혼동행렬 → 오류 중심 혼동행렬** 순서로 표시한다. 두 그림 모두 score ≥ 0.30, IoU ≥ 0.75 기준이며, 아래에 별도 표는 두지 않는다.

- **전체 혼동행렬:** 전체 56개 클래스와 미매칭 항목을 포함한다. 맞춘 객체는 대각선에 표시하고, 다른 클래스로 혼동한 객체와 미매칭 객체도 함께 보여준다.
- **오류 중심 혼동행렬:** 정답 대각선을 제외하고 오류가 있는 클래스만 확대한다. 클래스 혼동·미매칭 GT·추가 예측을 확인한다.
- 각 그림의 왼쪽은 건수, 오른쪽은 해당 클래스의 **전체 GT 객체 수** 대비 비율이다. 오류 중심 그림도 오류 건수만을 분모로 다시 정규화하지 않는다.
- 마지막 열의 미매칭은 연결되지 않은 GT, 마지막 행의 미매칭은 연결되지 않은 예측을 뜻한다. 미매칭 행과 GT가 없는 클래스 행의 비율은 정의하지 않아 비워 둔다.


In [ ]:
def plot_full_confusion(seed):
    """전체 클래스를 정답 대각선과 미매칭 항목까지 포함해 보여준다."""
    matrix = DIAGNOSTICS[seed]["matrix"]
    labels = [kcode(category) for category in CLASS_IDS] + ["미매칭"]
    denominators = matrix[:-1, :].sum(axis=1)
    normalized = np.full(matrix.shape, np.nan, dtype=float)
    np.divide(matrix[:-1, :], denominators[:, None], out=normalized[:-1, :],
              where=denominators[:, None] > 0)
    side = max(12, len(labels) * 0.29)
    fig, axes = plt.subplots(1, 2, figsize=(2 * side, side))
    for ax, values, ratio in [(axes[0], matrix, False), (axes[1], normalized, True)]:
        upper = 1 if ratio else max(1, int(matrix.max()))
        im = ax.imshow(np.ma.masked_invalid(values), cmap="Blues", vmin=0, vmax=upper)
        ax.set_xticks(range(len(labels)), labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(labels)), labels, fontsize=7)
        title = "전체 GT 대비 비율" if ratio else "전체 건수"
        ax.set(xlabel="예측 클래스", ylabel="정답 클래스",
               title=f"seed {seed} · 전체 혼동행렬 · {title} (정답 대각선 포함)\nscore ≥ {DIAGNOSTIC_SCORE:.2f}, IoU ≥ {DIAGNOSTIC_IOU:.2f}")
        for i, j in np.argwhere(np.isfinite(values) & (values > 0)):
            value = values[i, j]
            label = (f"{100 * value:.1f}".rstrip("0").rstrip(".") + "%") if ratio else str(int(value))
            ax.text(j, i, label, ha="center", va="center", fontsize=6,
                    color="white" if value > upper * 0.6 else "#17365d")
        fig.colorbar(im, ax=ax, fraction=0.035)
    show_figure(fig, f"03_seed{seed}_full_confusion_score030")


def plot_error_confusion(seed):
    """정답 대각선을 제외하고 오류가 있는 클래스만 보여준다."""
    original = DIAGNOSTICS[seed]["matrix"]
    bg = len(CLASS_IDS)
    errors = original.copy()
    np.fill_diagonal(errors, 0)
    active = np.flatnonzero((errors.sum(axis=0) + errors.sum(axis=1)) > 0).tolist()
    if not active:
        display(Markdown(f"seed {seed}: score {DIAGNOSTIC_SCORE:.2f}, IoU {DIAGNOSTIC_IOU:.2f} 기준 오류가 없습니다."))
        return
    small = errors[np.ix_(active, active)]
    denominators = original[:-1, :].sum(axis=1)
    normalized = np.full(errors.shape, np.nan, dtype=float)
    np.divide(errors[:-1, :], denominators[:, None], out=normalized[:-1, :], where=denominators[:, None] > 0)
    normalized = normalized[np.ix_(active, active)]
    labels = [kcode(CLASS_IDS[i]) if i < bg else "미매칭" for i in active]
    side = max(6.5, len(active) * 0.47)
    fig, axes = plt.subplots(1, 2, figsize=(2 * side, side))
    for ax, values, title in [(axes[0], small, "오류 건수"),
                              (axes[1], normalized, "전체 GT 대비 오류 비율")]:
        upper = max(1, int(small.max())) if title == "오류 건수" else 1
        im = ax.imshow(np.ma.masked_invalid(values), cmap="Reds", vmin=0, vmax=upper)
        ax.set_xticks(range(len(labels)), labels, rotation=90, fontsize=9)
        ax.set_yticks(range(len(labels)), labels, fontsize=9)
        ax.set(xlabel="예측 클래스", ylabel="정답 클래스",
               title=f"seed {seed} · {title} (정답 대각선 제외)\nscore ≥ {DIAGNOSTIC_SCORE:.2f}, IoU ≥ {DIAGNOSTIC_IOU:.2f}")
        for i, j in np.argwhere(np.isfinite(values) & (values > 0)):
            value = values[i, j]
            label = str(int(value)) if title == "오류 건수" else f"{100 * value:.1f}".rstrip("0").rstrip(".") + "%"
            ax.text(j, i, label, ha="center", va="center", fontsize=8,
                    color="white" if value > upper * 0.6 else "#9b1c1c")
        fig.colorbar(im, ax=ax, fraction=0.035)
    show_figure(fig, f"03_seed{seed}_error_confusion_score030")



for seed in SEEDS:
    display(Markdown(f"#### Seed {seed}: 전체 혼동행렬"))
    plot_full_confusion(seed)
    display(Markdown(f"#### Seed {seed}: 오류 중심 혼동행렬"))
    plot_error_confusion(seed)


### 3.3 어떤 클래스를 무엇으로 혼동했는가?

A→B와 B→A는 서로 다른 방향별 쌍이다. 먼저 전체 혼동 쌍을 정리하고, 이어서 **클래스 혼동이 실제 발생한 validation 이미지 전부**를 seed별로 확인한다. 같은 이미지에 여러 혼동 객체가 있으면 한 번에 묶어 표시한다.

- 기준: 각 seed의 best checkpoint, score ≥ 0.30, IoU ≥ 0.75.
- 위쪽: 전체 이미지의 GT 박스(초록)와 예측 박스(빨강)를 각각 표시.
- 아래쪽: 혼동한 객체마다 GT·예측 박스를 겹친 확대 이미지.
- 캡션: 정답→예측 클래스, score, IoU, seed, best epoch, 파일명.
- 이미지는 모두 validation이며 GT가 있다. 학습 중 W&B 표본이 아니라 분석에 사용한 best 예측 캐시를 연결한다.


In [ ]:
RECORD_INDEX = {seed: {r["file_name"]: r for r in records} for seed, records in RECORDS.items()}
_IMAGE_INDEX = None


def resolve_image(record):
    """원본 이름과 크기를 유지하는 로컬 이미지 경로를 찾는다."""
    global _IMAGE_INDEX
    name = record["file_name"]
    if name in IMAGE_PATH_OVERRIDES:
        candidate = Path(IMAGE_PATH_OVERRIDES[name])
        if not candidate.is_file():
            raise FileNotFoundError(f"직접 지정한 이미지가 없습니다: {candidate}")
        return candidate
    for candidate in [Path(record.get("image_path", "/__unavailable__")),
                      *(root / name for root in IMAGE_ROOTS)]:
        if candidate.is_file():
            return candidate
    if _IMAGE_INDEX is None:
        _IMAGE_INDEX = {}
        for root in IMAGE_ROOTS:
            if root.is_dir():
                for candidate in root.rglob("*"):
                    if candidate.is_file() and candidate.suffix.lower() in {".png", ".jpg", ".jpeg"}:
                        _IMAGE_INDEX.setdefault(candidate.name, set()).add(candidate.resolve())
    matches = sorted(_IMAGE_INDEX.get(name, set()))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"동일 파일명 이미지가 여러 개입니다. IMAGE_PATH_OVERRIDES로 지정하세요: {name}")
    return None


def draw_xyxy(ax, boxes, labels, color, scores=None, highlight=None):
    """기존 xyxy 좌표를 변환 없이 Matplotlib 사각형으로 표시한다."""
    used_labels = []
    line_height = max(20.0, abs(ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.035)
    label_width_per_char = abs(ax.get_xlim()[1] - ax.get_xlim()[0]) * 0.012
    for index, (box, category) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = map(float, box)
        width = 3 if highlight is not None and index == highlight else 1.5
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=width))
        text = kcode(category)
        if scores is not None:
            text += f" {float(scores[index]):.2f}"
        label_y = max(0, y1 - 5)
        label_width = len(text) * label_width_per_char
        for attempt in range(20):
            collision = any(x1 < other_x + other_width and x1 + label_width > other_x
                            and abs(label_y - other_y) < line_height
                            for other_x, other_y, other_width in used_labels)
            if not collision:
                break
            candidate_y = max(0, y1 - 5) - (attempt + 1) * line_height
            label_y = candidate_y if candidate_y >= line_height / 2 else y1 + (attempt + 1) * line_height
        used_labels.append((x1, label_y, label_width))
        ax.text(x1, label_y, text, color=color, fontsize=8,
                bbox={"facecolor": "white", "alpha": 0.75, "edgecolor": "none", "pad": 1}, clip_on=True)


def render_case(seed, event, figure_name):
    """validation 전체·확대 이미지와 수치 캡션을 화면에 보여준다."""
    record = RECORD_INDEX[seed][event["file_name"]]
    source = resolve_image(record)
    if source is None:
        display(Markdown(f"이미지 경로를 확인하세요: **{record['file_name']}**. IMAGE_ROOTS를 수정하면 저장 예측으로 표시할 수 있습니다."))
        return False
    with Image.open(source) as original:
        image = original.convert("RGB")
    expected_size = (int(record["width"]), int(record["height"]))
    if image.size != expected_size:
        raise ValueError(f"{source.name}: 원본 이미지 크기 {image.size}가 캐시 {expected_size}와 다릅니다. 좌표를 임의로 늘리지 않습니다.")
    target, prediction = record["target"], record["prediction"]
    keep = torch.where(prediction["scores"] >= DIAGNOSTIC_SCORE)[0]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax in axes:
        ax.imshow(image)
        ax.axis("off")
    draw_xyxy(axes[0], target["boxes"], target["labels"], "#159447")
    draw_xyxy(axes[1], prediction["boxes"][keep], prediction["labels"][keep], "#d62728", prediction["scores"][keep])
    draw_xyxy(axes[2], target["boxes"], target["labels"], "#159447")
    draw_xyxy(axes[2], prediction["boxes"][keep], prediction["labels"][keep], "#d62728", prediction["scores"][keep])
    g, p = event.get("gt_index"), event.get("pred_index")
    g = None if g is None or pd.isna(g) else int(g)
    p = None if p is None or pd.isna(p) else int(p)
    focus = target["boxes"][g] if g is not None else (prediction["boxes"][p] if p is not None else None)
    if focus is not None:
        x1, y1, x2, y2 = map(float, focus)
        pad = max(x2-x1, y2-y1) * 0.45 + 10
        axes[2].set_xlim(max(0, x1-pad), min(image.width, x2+pad))
        axes[2].set_ylim(min(image.height, y2+pad), max(0, y1-pad))
    axes[0].set_title("GT · 초록")
    axes[1].set_title(f"예측 · 빨강 · score ≥ {DIAGNOSTIC_SCORE:.2f}")
    axes[2].set_title("GT·예측 확대")
    kind = GT_KINDS.get(event["kind"], PRED_KINDS.get(event["kind"], event["kind"]))
    epoch = RUNS[seed]["result"]["best_epoch"]
    fig.suptitle(f"Validation / GT 있음 / seed {seed} best epoch {epoch}\n{record['file_name']} · {kind}")
    show_figure(fig, figure_name)
    true_category = event.get("true_category")
    pred_category = event.get("pred_category")
    caption = {"Seed": seed, "Checkpoint epoch": epoch, "데이터": "validation / GT 있음",
               "파일": record["file_name"], "진단 기준": f"score {DIAGNOSTIC_SCORE:.2f} / IoU {DIAGNOSTIC_IOU:.2f}",
               "유형": kind, "score": event.get("score"), "IoU": event.get("IoU")}
    for label, category in [("GT", true_category), ("예측", pred_category)]:
        caption[label] = "—" if category is None or pd.isna(category) else f"{kcode(category)} {CLASS_NAMES.get(int(category), '')}"
    show_table(pd.DataFrame([caption]).round(6))
    return True

In [ ]:
ALL_PAIRS = pd.concat([DIAGNOSTICS[s]["pairs"] for s in SEEDS], ignore_index=True)
for seed in SEEDS:
    show_table(DIAGNOSTICS[seed]["pairs"].round(6), f"seed {seed}: 전체 방향별 혼동 쌍")
if not ALL_PAIRS.empty:
    pair_recurrence = ALL_PAIRS.groupby(["정답 코드", "정답 클래스", "예측 코드", "예측 클래스"], as_index=False).agg(
        발생_seed수=("Seed", "nunique"), 합계_건수=("혼동 건수", "sum"),
        seed_목록=("Seed", lambda s: ", ".join(map(str, sorted(s))))
    ).sort_values(["발생_seed수", "합계_건수"], ascending=False)
    show_table(pair_recurrence, "Seed 사이 반복되는 혼동 쌍 (같은 GT의 반복 평가이므로 합계는 독립 객체 수가 아님)")
else:
    display(Markdown("이 기준에서는 방향별 클래스 혼동이 없습니다. 누락·위치 오류까지 없다는 뜻은 아닙니다."))


def render_confusion_image(seed, file_name, events):
    """같은 이미지의 실제 클래스 혼동을 GT·예측·객체별 확대로 표시한다."""
    record = RECORD_INDEX[seed][file_name]
    source = resolve_image(record)
    if source is None:
        raise FileNotFoundError(f"혼동 이미지가 없습니다. IMAGE_ROOTS를 확인하세요: {file_name}")
    with Image.open(source) as original:
        image = original.convert("RGB")
    if image.size != (int(record["width"]), int(record["height"])):
        raise ValueError(f"이미지 크기와 캐시 좌표 기준이 다릅니다: {file_name}")
    target, prediction = record["target"], record["prediction"]
    keep = torch.where(prediction["scores"] >= DIAGNOSTIC_SCORE)[0]
    detail_rows = math.ceil(len(events) / 2)
    fig = plt.figure(figsize=(15, 5.5 + 4.0 * detail_rows))
    grid = fig.add_gridspec(1 + detail_rows, 2, height_ratios=[1.4] + [1] * detail_rows)
    gt_ax, pred_ax = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])
    for ax in (gt_ax, pred_ax):
        ax.imshow(image)
        ax.axis("off")
    draw_xyxy(gt_ax, target["boxes"], target["labels"], "#159447")
    draw_xyxy(pred_ax, prediction["boxes"][keep], prediction["labels"][keep], "#d62728", prediction["scores"][keep])
    gt_ax.set_title("정답 GT 박스 · 초록")
    pred_ax.set_title(f"예측 박스 · 빨강 · score ≥ {DIAGNOSTIC_SCORE:.2f}")
    captions = []
    for index, event in enumerate(events.itertuples(index=False)):
        g, p = int(event.gt_index), int(event.pred_index)
        if int(target["labels"][g]) == int(prediction["labels"][p]):
            raise ValueError("클래스가 같은 객체가 혼동 사례에 들어갔습니다.")
        if float(prediction["scores"][p]) < DIAGNOSTIC_SCORE or float(event.IoU) < DIAGNOSTIC_IOU:
            raise ValueError("혼동 사례의 score·IoU 기준이 다릅니다.")
        ax = fig.add_subplot(grid[1 + index // 2, index % 2])
        ax.imshow(image)
        boxes = [target["boxes"][g], prediction["boxes"][p]]
        for box, color in zip(boxes, ("#159447", "#d62728")):
            x1, y1, x2, y2 = map(float, box)
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2.5))
        xmin = min(float(box[0]) for box in boxes)
        ymin = min(float(box[1]) for box in boxes)
        xmax = max(float(box[2]) for box in boxes)
        ymax = max(float(box[3]) for box in boxes)
        pad = max(xmax-xmin, ymax-ymin) * 0.28 + 10
        ax.set_xlim(max(0, xmin-pad), min(image.width, xmax+pad))
        ax.set_ylim(min(image.height, ymax+pad), max(0, ymin-pad))
        ax.axis("off")
        true_category, pred_category = int(event.true_category), int(event.pred_category)
        ax.set_title(f"{index+1}. {kcode(true_category)} → {kcode(pred_category)}\nscore {event.score:.3f} · IoU {event.IoU:.3f}")
        captions.append(f"{index+1}. **{kcode(true_category)} {CLASS_NAMES[true_category]} → "
                        f"{kcode(pred_category)} {CLASS_NAMES[pred_category]}** · score {event.score:.4f} · IoU {event.IoU:.4f}")
    epoch = RUNS[seed]["result"]["best_epoch"]
    fig.suptitle(f"Validation / GT 있음 / seed {seed} best epoch {epoch}\n{file_name} · 실제 클래스 혼동 {len(events)}건", y=1.01)
    image_number = len(CONFUSION_IMAGE_AUDIT) + 1
    show_figure(fig, f"03_seed{seed}_class_confusion_{image_number:02d}_score030")
    display(Markdown("\n\n".join(captions)))
    CONFUSION_IMAGE_AUDIT.append({"Seed": seed, "file_name": file_name, "혼동 GT 건수": len(events)})


CONFUSION_IMAGE_AUDIT = []
for seed in SEEDS:
    events = DIAGNOSTICS[seed]["gt_events"]
    confused = events.loc[events["kind"].eq("class_confusion")].sort_values(["file_name", "gt_index"])
    if confused.empty:
        display(Markdown(f"seed {seed}: 이 기준의 클래스 혼동이 없습니다."))
    for file_name, group in confused.groupby("file_name", sort=True):
        render_confusion_image(seed, file_name, group)
    rendered = sum(row["혼동 GT 건수"] for row in CONFUSION_IMAGE_AUDIT if row["Seed"] == seed)
    if rendered != len(confused) or rendered != int(DIAGNOSTICS[seed]["pairs"]["혼동 건수"].sum()):
        raise ValueError(f"seed {seed}: 혼동 쌍 표와 표시한 이미지의 오류 객체 수가 다릅니다.")


## 4. 박스 분석: 위치를 얼마나 정밀하게 맞췄는가?

### 4.1 전체 클래스의 IoU별 AP와 학습 빈도

Seed마다 한 장에 validation 정답이 있는 **30개 클래스 전부**를 표시한다. 세 seed의 행 순서는 seed 0의 평균 AP 오름차순으로 통일해 비교하기 쉽게 한다.

- 왼쪽 파랑 막대: 평균 AP, 아래 축 0~1.
- 왼쪽 주황 막대: train의 정답 객체 수, 위 축 '개'. 반복 학습 횟수가 아니다.
- 오른쪽: 같은 클래스 순서로 다섯 IoU별 AP 점. 색과 모양으로 구분한다.
- Validation 정답이 없는 26개 클래스는 1.1의 평가 불가 표에 표시한다. 0점으로 넣지 않는다.

AP는 클래스와 위치를 함께 평가하므로 AP75−AP95는 **정밀한 위치를 확인할 단서**이며 위치 오류의 확정 건수가 아니다.

In [ ]:
CLASS_ORDER = AP_TABLES[SEEDS[0]].sort_values(["AP75_95", "category_id"])["category_id"].tolist()
expected_classes = {c for c, n in VAL_FREQUENCY.items() if n > 0}
if set(CLASS_ORDER) != expected_classes:
    raise ValueError("클래스별 AP 목록이 validation 정답 클래스와 다릅니다.")
frequency_max = max(FREQUENCY[c] for c in CLASS_ORDER)
markers = ["o", "s", "^", "D", "X"]
for seed in SEEDS:
    table = AP_TABLES[seed].set_index("category_id").loc[CLASS_ORDER].reset_index()
    fig, axes = plt.subplots(1, 2, sharey=True, figsize=(18, max(11, len(table) * 0.43 + 2.8)),
                             gridspec_kw={"width_ratios": [1, 1.1]})
    y = np.arange(len(table))
    axes[0].barh(y, table["AP75_95"], height=0.70, color="#2563eb", alpha=0.85, label="평균 AP")
    axes[0].set_xlim(0, 1.12)
    axes[0].set_yticks(y, [f"{r.k_code}\n{r.name}" for r in table.itertuples()], fontsize=8)
    axes[0].invert_yaxis()
    axes[0].set_xlabel("평균 AP@[0.75:0.05:0.95] (아래 축)")
    frequency_axis = axes[0].twiny()
    frequency_axis.patch.set_visible(False)
    frequency_axis.set_xlim(0, max(1, frequency_max) * 1.12)
    frequency_axis.barh(y, table["train_objects"], height=0.22, color="#e29024", alpha=0.65)
    frequency_axis.set_xlabel("Train 정답 객체 수 (개, 위 축)", color="#9a570d")
    frequency_axis.tick_params(axis="x", colors="#9a570d")
    frequency_axis.tick_params(axis="y", left=False, labelleft=False)
    for position, row in enumerate(table.itertuples()):
        axes[0].annotate(f"{row.AP75_95:.4f}", (row.AP75_95, position - 0.23),
                         xytext=(4, 0), textcoords="offset points", fontsize=8, color="#1d4f9b")
        frequency_axis.annotate(str(row.train_objects), (row.train_objects, position + 0.24),
                                xytext=(4, 0), textcoords="offset points", fontsize=7, color="#9a570d")
    for index, iou in enumerate(AP_IOUS):
        axes[1].scatter(table[f"AP{round(iou * 100)}"], y + (index - 2) * 0.10,
                        color=IOU_COLORS[index], marker=markers[index], s=28, label=f"IoU {iou:.2f}")
    axes[1].set(xlim=(-0.02, 1.04), xlabel="AP (0~1)", title="전체 클래스의 IoU별 AP")
    axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=5, fontsize=8)
    for ax in axes:
        ax.grid(axis="x", alpha=0.15)
        ax.set_axisbelow(True)
    fig.suptitle(f"1차 seed {seed} · best epoch {RUNS[seed]['result']['best_epoch']} · validation 47장", y=1.015)
    show_figure(fig, f"04_seed{seed}_all_class_ap_frequency")
    show_table(table[["k_code", "name", "train_objects", "GT_objects", "AP75", "AP80", "AP85", "AP90", "AP95", "AP75_95"]].round(6),
               f"seed {seed}: 전체 클래스 AP와 객체 수")

### 4.2 엄격한 IoU에서 성능이 떨어지는 클래스

전체 클래스의 AP75−AP95 차이를 확인한다. 이어서 **클래스를 무시하고** score 0.30 예측과 GT를 각 IoU에서 일대일로 연결해, 위치 매칭 GT 비율을 본다. 이 비율은 AP도 클래스별 recall도 아니다.

이를 통해 클래스 구분의 영향을 줄여 위치 겹침을 살펴볼 수 있다. 다만 일대일 greedy 매칭과 score 선택에 영향을 받는다.

In [ ]:
for seed in SEEDS:
    gap = AP_TABLES[seed].sort_values(["AP75_minus_AP95", "category_id"], ascending=[False, True])
    show_table(gap[["k_code", "name", "GT_objects", "AP75", "AP95", "AP75_minus_AP95"]].round(6),
               f"seed {seed}: 모든 클래스의 엄격한 IoU 성능 차이")
geometry_rows = []
for seed in SEEDS:
    for iou in AP_IOUS:
        matched, correct_class = 0, 0
        for record in RECORDS[seed]:
            target, prediction = record["target"], record["prediction"]
            keep = torch.where(prediction["scores"] >= DIAGNOSTIC_SCORE)[0]
            overlaps = box_iou(target["boxes"], prediction["boxes"][keep]).numpy()
            pairs = greedy_pairs(overlaps, range(len(target["labels"])), range(len(keep)), iou)
            matched += len(pairs)
            correct_class += sum(int(target["labels"][g]) == int(prediction["labels"][keep[p]]) for g, p, _ in pairs)
        total = int(SUPPORT["val"].sum())
        geometry_rows.append({"Seed": seed, "IoU": iou, "GT 객체": total,
                              "클래스 무관 위치 매칭": matched,
                              "그중 클래스도 정답": correct_class,
                              "위치 매칭 비율": matched / total})
GEOMETRY = pd.DataFrame(geometry_rows)
show_table(GEOMETRY.round(6), "score 0.30, 클래스 무관 위치 매칭")
fig, ax = plt.subplots(figsize=(9, 4.5))
for seed in SEEDS:
    rows = GEOMETRY.loc[GEOMETRY["Seed"].eq(seed)]
    ax.plot(rows["IoU"], rows["위치 매칭 비율"], marker="o", color=SEED_COLORS[seed], label=f"seed {seed}")
ax.set(xticks=AP_IOUS, xlabel="박스 매칭 IoU 기준", ylabel="전체 GT 중 위치가 매칭된 비율", ylim=(0, 1.02),
       title="클래스 구분을 제외한 위치 매칭 · score ≥ 0.30")
ax.grid(alpha=0.18)
ax.legend()
show_figure(fig, "04_geometry_matching")

### 4.3 실제 GT·예측 박스 비교

Best 캐시와 **동일 파일명의 원본 validation 이미지**를 연결한다. 저장된 좌표는 원본 픽셀의 xyxy이며 재학습·재추론하지 않는다. 이미지가 없거나 크기가 다르면 표시하지 않고 경로 확인이 필요하다고 알린다.

왼쪽 GT, 가운데 score 0.30 이상의 예측, 오른쪽 해당 GT 주변 확대 영역을 보여준다. 이미지는 validation이며 GT가 있다. W&B 학습 중 표본을 best 오류 사례로 대체하지 않는다.

In [ ]:
# 클래스가 맞은 고IoU 매칭 중, IoU가 가장 낮은 사례를 seed마다 한 장 확인한다.
for seed in SEEDS:
    candidates = DIAGNOSTICS[seed]["gt_events"].query("kind == 'correct'").sort_values(["IoU", "file_name"])
    if not candidates.empty:
        event = candidates.iloc[0].to_dict()
        render_case(seed, event, f"04_seed{seed}_matched_box_detail")

## 5. 오류 분석: 못 맞춘 사례는 무엇이었는가?

### 5.1 오류 사례 목록

혼동행렬에서 미매칭으로 남은 GT는 저장 예측과 다시 연결해 **사례 탐색용 후보**를 정리한다. 남은 예측 중 IoU가 0보다 크고 0.75 미만인 박스를 클래스와 무관하게 일대일 연결한다. 이는 오류 이미지를 찾는 보조 규칙이며 AP·혼동행렬 점수를 바꾸지 않는다.

- **클래스 혼동:** IoU ≥ 0.75로 연결됐지만 클래스가 다름.
- **위치 불일치 후보:** 남은 박스가 일부 겹치고 클래스는 같음.
- **위치·클래스 복합 후보:** 남은 박스가 일부 겹치고 클래스도 다름.
- **미매칭 GT·원인 미분류:** 위 연결에도 대응 예측이 없음. score 때문에 제외됐는지 등은 이미지로 추가 확인.
- **중복·추가 예측 후보:** 위 후보 연결에 사용되지 않은 남은 예측. GT 오류 건수와 합산하지 않음.

GT별 목록은 정답 객체마다 정확히 한 행이다. 하나의 오탐·미탐이 같은 문제에서 함께 생길 수 있으므로 FP+FN이나 GT·예측 이벤트 합을 '틀린 알약 수'로 해석하지 않는다.

In [ ]:
ALL_GT_EVENTS = pd.concat([DIAGNOSTICS[s]["gt_events"] for s in SEEDS], ignore_index=True)
GT_ERRORS = ALL_GT_EVENTS.loc[ALL_GT_EVENTS["kind"].ne("correct")].copy()
PRED_ERRORS = pd.concat([DIAGNOSTICS[s]["pred_events"] for s in SEEDS], ignore_index=True)
for seed in SEEDS:
    errors = GT_ERRORS.loc[GT_ERRORS["Seed"].eq(seed)].copy()
    errors["오류 후보"] = errors["kind"].map(GT_KINDS)
    for field, label in [("true_category", "정답"), ("pred_category", "예측")]:
        errors[label] = errors[field].map(lambda c: "—" if pd.isna(c) else kcode(c))
    show_table(errors[["file_name", "gt_index", "오류 후보", "정답", "예측", "score", "IoU"]].round(6),
               f"seed {seed}: 전체 GT 오류 후보 목록")
    extra = PRED_ERRORS.loc[PRED_ERRORS["Seed"].eq(seed)].copy()
    if not extra.empty:
        extra["오류 후보"] = extra["kind"].map(PRED_KINDS)
        extra["예측"] = extra["pred_category"].map(kcode)
        show_table(extra[["file_name", "pred_index", "오류 후보", "예측", "score", "IoU"]].round(6),
                   f"seed {seed}: 별도 중복·추가 예측 후보")
kind_rows = []
for seed in SEEDS:
    counts_by_kind = DIAGNOSTICS[seed]["gt_events"]["kind"].value_counts()
    for kind, label in GT_KINDS.items():
        kind_rows.append({"Seed": seed, "유형": label, "GT 객체": int(counts_by_kind.get(kind, 0))})
GT_KIND_COUNTS = pd.DataFrame(kind_rows)
show_table(GT_KIND_COUNTS, "서로 중복되지 않는 GT 기준 분류 (정답 포함 합계=159/seed)")
fig, ax = plt.subplots(figsize=(12, 5))
error_kinds = [k for k in GT_KINDS if k != "correct"]
y = np.arange(len(error_kinds))
for offset, seed in zip((-0.23, 0, 0.23), SEEDS):
    values = DIAGNOSTICS[seed]["gt_events"]["kind"].value_counts()
    ax.barh(y + offset, [int(values.get(k, 0)) for k in error_kinds], height=0.22,
            color=SEED_COLORS[seed], label=f"seed {seed}")
ax.set_yticks(y, [GT_KINDS[k] for k in error_kinds])
ax.invert_yaxis()
ax.set(xlabel="GT 객체 수 (개, 후보 분류)", title="score 0.30 · GT 기준 오류 양상")
ax.legend()
show_figure(fig, "05_gt_error_candidates")

### 5.2 대표 오류 이미지

클래스 혼동 이미지는 3.3에 모두 표시했다. 여기서는 각 seed의 나머지 오류 후보 유형마다 한 사례를 선택한다. 전체 목록은 5.1에 남기고, 이미지 수를 늘리고 싶을 때만 아래 MAX_CASES_PER_KIND를 조정한다. 고IoU 클래스 혼동은 score가 높은 사례부터, 위치 후보는 IoU가 큰 사례부터 보여준다.

동일 GT가 세 seed에서 반복적으로 틀렸는지도 함께 확인한다. 서로 다른 seed의 같은 알약은 독립적인 새 객체 3개가 아니다.

In [ ]:
MAX_CASES_PER_KIND = 1
for seed in SEEDS:
    for kind in [k for k in GT_KINDS if k not in {"correct", "class_confusion"}]:
        candidates = GT_ERRORS.loc[GT_ERRORS["Seed"].eq(seed) & GT_ERRORS["kind"].eq(kind)]
        column = "score" if kind == "class_confusion" else "IoU"
        candidates = candidates.sort_values([column, "file_name"], ascending=[False, True], na_position="last")
        for number, (_, event) in enumerate(candidates.head(MAX_CASES_PER_KIND).iterrows(), start=1):
            render_case(seed, event.to_dict(), f"05_seed{seed}_{kind}_{number}")
    for kind in PRED_KINDS:
        candidates = PRED_ERRORS.loc[PRED_ERRORS["Seed"].eq(seed) & PRED_ERRORS["kind"].eq(kind)]
        for number, (_, event) in enumerate(candidates.sort_values("score", ascending=False).head(MAX_CASES_PER_KIND).iterrows(), start=1):
            render_case(seed, event.to_dict(), f"05_seed{seed}_{kind}_{number}")

if not GT_ERRORS.empty:
    repeated_gt = GT_ERRORS.groupby(["file_name", "gt_index", "true_category"], as_index=False).agg(
        오류_seed수=("Seed", "nunique"), seed_목록=("Seed", lambda s: ", ".join(map(str, sorted(s)))),
        후보_유형=("kind", lambda ks: ", ".join(sorted({GT_KINDS[k] for k in ks})))
    ).sort_values(["오류_seed수", "file_name"], ascending=[False, True])
    repeated_gt["정답 코드"] = repeated_gt["true_category"].map(kcode)
    show_table(repeated_gt.drop(columns="true_category"), "세 seed의 동일 GT 오류 반복 여부")
else:
    display(Markdown("이 진단 기준의 GT 오류 후보는 없습니다. 엄격한 IoU의 AP와 함께 해석하세요."))

### 5.3 박스와 클래스 중 어떤 문제가 두드러지는가?

5.1의 **GT 기준**에서 클래스 혼동·위치 후보·복합 후보·미분류 건수를 비교한다. 복합 후보를 분류 또는 위치 한쪽에 억지로 넣지 않는다. score가 낮아 제외된 정답이나 다른 박스에 먼저 연결된 경우도 있으므로 이 비율을 모델의 오류 원인 비중으로 확정하지 않는다.

**작성할 해석:**
1. 세 seed에서 공통으로 나타난 오류 양상은 무엇인가?
2. IoU별 AP 변화와 클래스 무관 위치 매칭 결과가 실제 박스 사례와 일치하는가?
3. 클래스가 같은데 박스만 어긋난 사례, 위치는 충분히 맞는데 클래스가 다른 사례가 각각 있는가?
4. 판단하기 어려운 복합 사례는 무엇인가?

## 6. 정리 및 해석

### 6.1 확인된 사실

아래 요약은 저장 로그·best 캐시의 계산 결과를 한곳에 모은다. 0.30 결과를 실행하기 전에는 오류 숫자에 관한 결론을 미리 채우지 않는다.

In [ ]:
summary = BEST_TABLE[["Seed", "Best epoch", "Best validation mAP"]].merge(DIAGNOSTIC_TABLE, on="Seed")
summary["방향별 혼동 쌍 수"] = summary["Seed"].map({s: len(DIAGNOSTICS[s]["pairs"]) for s in SEEDS})
summary["클래스 혼동 GT 건수"] = summary["Seed"].map({s: int(DIAGNOSTICS[s]["gt_events"]["kind"].eq("class_confusion").sum()) for s in SEEDS})
show_table(summary.round(6), "Seed별 best 성능과 score 0.30 진단 요약")
ap_comparison = SUPPORT[["category_id", "name", "train", "val"]].copy()
for seed in SEEDS:
    ap_comparison = ap_comparison.merge(AP_TABLES[seed][["category_id", "AP75_95"]].rename(columns={"AP75_95": f"seed{seed}_AP"}),
                                         on="category_id", how="left", validate="one_to_one")
ap_columns = [f"seed{seed}_AP" for seed in SEEDS]
ap_comparison["AP 평균"] = ap_comparison[ap_columns].mean(axis=1)
ap_comparison["AP 표본 표준편차"] = ap_comparison[ap_columns].std(axis=1, ddof=1)
ap_comparison.insert(1, "k_code", ap_comparison["category_id"].map(kcode))
show_table(ap_comparison.sort_values(["AP 평균", "category_id"], na_position="last").round(6),
           "모든 학습 클래스: seed별 AP와 변동 (평가 불가 클래스는 빈칸)")

### 6.2 데이터 분포와 성능의 관계

학습 빈도가 적은 클래스가 일관되게 낮은지, 같은 빈도에서도 성능이 다른지 본다. 아래 순위 상관계수는 각 seed에서 평가 가능한 30개 클래스의 관계 요약이다. 상관은 인과관계가 아니며 클래스별 GT 수·외형·촬영 조건의 영향을 분리하지 못한다.

In [ ]:
correlation_rows = []
for seed in SEEDS:
    table = AP_TABLES[seed]
    rank_correlation = table["train_objects"].rank().corr(table["AP75_95"].rank())
    correlation_rows.append({"Seed": seed, "클래스 수": len(table),
                             "학습 객체 수와 AP의 순위 상관": rank_correlation})
show_table(pd.DataFrame(correlation_rows).round(6))
fig, axes = plt.subplots(1, len(SEEDS), figsize=(16, 4.8), sharex=True, sharey=True)
for ax, seed in zip(axes, SEEDS):
    table = AP_TABLES[seed]
    ax.scatter(table["train_objects"], table["AP75_95"], color=SEED_COLORS[seed], s=35, alpha=0.8)
    ax.set(xlabel="Train 정답 객체 수 (개)", ylabel="평균 AP@[.75:.95]", title=f"seed {seed}", ylim=(0, 1.05))
    ax.grid(alpha=0.18)
show_figure(fig, "06_frequency_ap_relationship")

### 6.3 실제 Kaggle 제출 결과와 validation 비교

1차 실험의 세 seed를 실제로 제출했다. 아래 점수는 **사용자가 제공한 Kaggle 제출 완료 화면**에서 확인했으며, 제출 CSV 파일명으로 해당 seed/run을 연결했다.

| Seed | 실제 Kaggle 제출 점수 | 제출 run 식별자 |
|---|---:|---|
| 0 | 0.37633 | 88800b754961 |
| 1 | 0.37750 | 91b0ad1f9226 |
| 2 | 0.39476 | b013ae09ae90 |

아래 코드는 제출 점수를 다시 계산하는 코드가 아니다. 확인된 실제 점수를 저장된 validation 결과와 나란히 표시한다. 제출을 새로 실행하지 않는다.


In [ ]:
# 근거: 사용자가 제공한 Kaggle 제출 완료 화면. 실제 점수는 로컬 scores.json에서 가져오지 않는다.
KAGGLE_SUBMISSIONS = {
    0: {"score": 0.37633, "file": "cj_faster_e100_s0_exp1_b16_fast_e100_ft_lr3e-4__88800b754961.csv"},
    1: {"score": 0.37750, "file": "cj_faster_e100_s1_exp1_b16_fast_e100_ft_lr3e-4__91b0ad1f9226.csv"},
    2: {"score": 0.39476, "file": "cj_faster_e100_s2_exp1_b16_fast_e100_ft_lr3e-4__b013ae09ae90.csv"},
}
kaggle_rows = []
for seed in SEEDS:
    submission = KAGGLE_SUBMISSIONS[seed]
    run = RUNS[seed]
    if submission["file"] != run["root"].name + ".csv":
        raise ValueError(f"seed {seed}: 제출 CSV와 분석 run의 파일명이 다릅니다.")
    validation = run["result"]["best_val_map_75_95"]
    kaggle_rows.append({
        "Seed": seed, "Best epoch": run["result"]["best_epoch"],
        "Validation mAP": validation, "실제 Kaggle 점수": submission["score"],
        "Validation − Kaggle (p.p.)": 100 * (validation - submission["score"]),
    })
KAGGLE_COMPARISON = pd.DataFrame(kaggle_rows)
show_table(KAGGLE_COMPARISON.round(6), "각 seed의 best validation과 실제 Kaggle 제출 결과")
show_table(pd.DataFrame([{"Seed": seed, "제출 CSV": item["file"]} for seed, item in KAGGLE_SUBMISSIONS.items()]),
           "제출 파일과 seed 대응 (근거: 사용자 제공 화면)")
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(KAGGLE_COMPARISON))
for shift, column, label, color in [
    (-0.18, "Validation mAP", "Validation mAP · 47장", "#2563eb"),
    (0.18, "실제 Kaggle 점수", "실제 Kaggle 제출 점수", "#d97706"),
]:
    values = KAGGLE_COMPARISON[column].to_numpy()
    bars = ax.bar(x + shift, values, width=0.34, color=color, label=label)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, value + 0.015, f"{value:.5f}", ha="center", fontsize=10)
ax.set(xticks=x, xticklabels=[f"seed {seed}" for seed in SEEDS], ylim=(0, 1.05),
       ylabel="mAP@[0.75:0.05:0.95]", title="1차 실험: validation과 실제 Kaggle 제출 결과")
ax.legend()
ax.grid(axis="y", alpha=0.18)
ax.set_axisbelow(True)
show_figure(fig, "06_kaggle_validation_comparison")


**관찰된 사실**
- Validation은 seed 0이 가장 높지만, 실제 Kaggle 점수는 seed 2가 **0.39476**으로 가장 높다. 이 세 run에서 validation의 순위와 Kaggle의 순위는 일치하지 않는다.
- Validation은 약 0.918~0.933, 실제 Kaggle은 0.37633~0.39476이다. 차이는 약 **52.31~55.69 p.p.**다. 이는 서로 다른 평가 데이터에서 관찰한 점수 차이이며 원인을 증명하는 수치는 아니다.

**해석의 범위와 다음 확인**
- 기존 validation 47장에는 정답 클래스가 30개만 있어 test 전체의 클래스·촬영 조건을 모두 평가하지 못한다.
- 초기 train에 없는 test 등장 18개 클래스는 격차의 가능한 원인이지만, 숨겨진 test GT가 없으므로 해당 클래스가 설명하는 손실량을 확정할 수 없다.
- 촬영 조건의 차이, 학습 데이터의 부족·불균형, 제출 좌표·클래스 매핑·후처리 조건은 검토 대상이다. 이 점수만으로 특정 원인이나 과적합 여부를 확정하지 않는다.
- 2차 실험과 비교할 때는 공통 validation 47장 비교와 실제 Kaggle 제출 점수 비교를 나누고, 데이터 구성·학습량 변화도 함께 설명한다.


### 6.4 1차 실험의 한계

- Train 185장, validation 47장이다. 일부 클래스의 GT 객체 수가 적어 소수 오류의 영향이 크다.
- 학습 클래스 56개 중 validation에서 직접 평가되는 클래스는 30개다.
- 초기 train에 없는 test 등장 18개 클래스는 이 모델이 학습하지 못했다.
- Train 빈도와 AP 관계만으로 데이터 불균형이 오류의 원인이라고 확정할 수 없다.
- 높은 validation mAP만으로 과소적합·과적합이 없다고 단정할 수 없다.
- Validation과 실제 Kaggle의 평가 데이터·클래스 범위가 다르므로, 두 점수의 차이만으로 격차의 원인을 확정할 수 없다.

### 6.5 2차 실험으로 이어지는 질문

1. 부족한 클래스와 미학습 클래스를 보충하면 공통 validation 47장에서 무엇이 달라지는가?
2. 데이터 구성 방법에 따라 클래스 혼동과 박스 위치 정밀도가 각각 개선되는가?
3. 이미지 수·클래스 범위·실제 학습 step이 함께 달라지는 영향을 어떻게 구분할 것인가?
4. Validation 개선이 실제 Kaggle 성능과 같은 방향으로 나타나는가?

### 노션 보고서로 옮길 해석 틀

| 구분 | 작성할 내용 |
|---|---|
| 확인된 사실 | 실행 결과의 수치와 해당 표·그림 번호 |
| 가능한 설명 | 실제 클래스·박스 사례가 뒷받침하는 해석 |
| 아직 검증하지 못한 가설 | 추가 데이터·분석 없이는 확정할 수 없는 설명 |
| 다음 실험 질문 | 2차 실험에서 비교할 항목과 기대하는 관찰 |

**그림 저장:** 필요한 그림을 확인한 뒤에만 SAVE_FIGURES를 True로 바꾼다. 기존 그림은 덮어쓰지 않으며, 입력 run 폴더에는 쓰지 않는다. 설정과 결과는 화면의 표를 노션으로 옮기고 CSV는 생성하지 않는다.

### 근거 자료
- 각 run의 config.json, result.json, metrics.csv, comparison_manifest.json, validation_predictions.pt
- 실험 그룹의 settings.json, class_support.csv
- data/datasets/classes.csv: 클래스 그룹·test 검토 추정치
- 04-cj-faster-rcnn-exp1.ipynb: 원래 입력 처리·스케줄러·평가·캐시 내보내기 구현 (실행하지 않음)
- RULE.md: 프로젝트 평가 지표
- 실제 Kaggle 제출 점수: 사용자가 제공한 제출 완료 화면과 CSV 파일명으로 seed별 대응 확인
